# LangChain Agent Benchmark 03: Architectures, Memory, and Tools

This notebook compares direct LLM calls, RAG-flavored calls, tool-using agents, LangGraph workflows, multi-agent patterns, routers, sequential chains, memory, and tool design using the local airway RNA-seq data in `data/` and the paper/readmes in `docs/` and `data/`.


In [ ]:
# Run once per environment. Keep optional provider packages commented until needed.
%pip install -qU langchain langchain-core langchain-community langchain-openai langchain-anthropic langchain-google-genai langchain-text-splitters langgraph pandas pydantic pypdf langchain-huggingface sentence-transformers
# Optional local/vector-store packages:
# %pip install -qU faiss-cpu langchain-chroma langchain-qdrant qdrant-client


## Set up environment and data

In [ ]:
import os
import sqlite3
from pathlib import Path
from typing import TypedDict

import pandas as pd

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_core.tools import tool
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import END, START, StateGraph

if not os.getenv("GOOGLE_API_KEY"):
    print("Set GOOGLE_API_KEY before running the examples.")

DATA_DIR = Path("data")
DOCS_DIR = Path("docs")

sample_metadata = pd.read_csv(DATA_DIR / "airway_metadata.csv").rename(columns={
    "Unnamed: 0": "sample_id",
    "SampleName": "sample_name",
    "Run": "run",
    "avgLength": "avg_length",
    "Experiment": "experiment",
    "Sample": "sra_sample",
    "BioSample": "biosample",
})

gene_counts = pd.read_csv(DATA_DIR / "airway_counts.csv").rename(columns={"Unnamed: 0": "row_id"})
gene_counts = gene_counts.drop(columns=["row_id"])
sample_columns = [column for column in gene_counts.columns if column.startswith("SRR")]

deseq2_results = pd.read_csv(DATA_DIR / "DESeq2_results_airway.csv").rename(columns={"Unnamed: 0": "gene_id"})
deseq2_results = deseq2_results.merge(
    gene_counts[["gene_id", "gene_name", "symbol", "gene_biotype", "seq_name"]].drop_duplicates("gene_id"),
    on="gene_id",
    how="left",
)

connection = sqlite3.connect(":memory:")
sample_metadata.to_sql("sample_metadata", connection, index=False, if_exists="replace")
gene_counts.to_sql("gene_counts", connection, index=False, if_exists="replace")
deseq2_results.to_sql("deseq2_results", connection, index=False, if_exists="replace")
connection.commit()

paper_docs = []
try:
    paper_docs = PyPDFLoader(str(DOCS_DIR / "paper.pdf")).load()
    for doc in paper_docs:
        doc.metadata["source"] = str(DOCS_DIR / "paper.pdf")
        doc.metadata["doc_type"] = "paper_pdf"
except Exception as exc:
    paper_docs = [Document(page_content=f"docs/paper.pdf could not be loaded: {exc}", metadata={"source": str(DOCS_DIR / "paper.pdf"), "doc_type": "paper_pdf_error"})]

markdown_docs = [
    Document(page_content=(DATA_DIR / "README.md").read_text(), metadata={"source": str(DATA_DIR / "README.md"), "doc_type": "markdown"}),
    Document(page_content=(DATA_DIR / "DESeq2_results_airway_readme.md").read_text(), metadata={"source": str(DATA_DIR / "DESeq2_results_airway_readme.md"), "doc_type": "markdown"}),
]

text_splitter = RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=150)
doc_chunks = text_splitter.split_documents(paper_docs + markdown_docs)

try:
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        encode_kwargs={"normalize_embeddings": True},
    )
    doc_vector_store = InMemoryVectorStore.from_documents(doc_chunks, embedding=embeddings)
    document_retrieval_mode = "semantic"
except Exception as exc:
    doc_vector_store = None
    document_retrieval_mode = f"lexical fallback because embeddings failed: {exc}"



## Create tools

In [ ]:
@tool
def query_sample_metadata(sql: str) -> str:
    """Run read-only SQL against sample_metadata. Columns: sample_id, sample_name, cell, dex, albut, run, avg_length, experiment, sra_sample, biosample. Use this for sample counts, treatment labels, and accessions. Example: select dex, count(*) as n from sample_metadata group by dex."""
    if not sql.strip().lower().startswith(("select", "with")):
        return "Only SELECT or WITH queries are allowed."
    try:
        result = pd.read_sql_query(sql, connection)
        if result.empty:
            return "Query returned no rows."
        return result.head(50).to_string(index=False)
    except Exception as exc:
        return f"SQL error: {exc}"

@tool
def query_deseq2_results(sql: str) -> str:
    """Run read-only SQL against deseq2_results. Columns: gene_id, baseMean, log2FoldChange, lfcSE, pvalue, padj, gene_name, symbol, gene_biotype, seq_name. Use this for differential expression statistics from dexamethasone treatment versus untreated control. Example: select gene_id, symbol, log2FoldChange, padj from deseq2_results where padj < 0.05 order by log2FoldChange desc limit 10."""
    if not sql.strip().lower().startswith(("select", "with")):
        return "Only SELECT or WITH queries are allowed."
    try:
        result = pd.read_sql_query(sql, connection)
        if result.empty:
            return "Query returned no rows."
        return result.head(50).to_string(index=False)
    except Exception as exc:
        return f"SQL error: {exc}"

@tool
def query_gene_counts(sql: str) -> str:
    """Run read-only SQL against gene_counts. Columns include gene_id, gene_name, entrezid, gene_biotype, gene_seq_start, gene_seq_end, seq_name, seq_strand, seq_coord_system, symbol, and raw count columns SRR1039508, SRR1039509, SRR1039512, SRR1039513, SRR1039516, SRR1039517, SRR1039520, SRR1039521. Use this for raw counts by gene or sample. Example: select gene_id, symbol, SRR1039508, SRR1039509 from gene_counts where symbol = 'CRISPLD2'."""
    if not sql.strip().lower().startswith(("select", "with")):
        return "Only SELECT or WITH queries are allowed."
    try:
        result = pd.read_sql_query(sql, connection)
        if result.empty:
            return "Query returned no rows."
        return result.head(50).to_string(index=False)
    except Exception as exc:
        return f"SQL error: {exc}"

@tool
def summarize_gene_counts(identifier: str) -> str:
    """Summarize raw airway count values for one gene by Ensembl gene_id, gene_name, or symbol, including per-sample counts and mean counts by dex treatment group."""
    matches = gene_counts[
        gene_counts["gene_id"].astype(str).str.casefold().eq(identifier.casefold())
        | gene_counts["gene_name"].astype(str).str.casefold().eq(identifier.casefold())
        | gene_counts["symbol"].astype(str).str.casefold().eq(identifier.casefold())
    ]
    if matches.empty:
        return f"No gene matched {identifier}."
    row = matches.iloc[0]
    per_sample = pd.DataFrame({"sample_id": sample_columns, "raw_count": [row[column] for column in sample_columns]})
    per_sample = per_sample.merge(sample_metadata[["sample_id", "dex", "cell"]], on="sample_id", how="left")
    means = per_sample.groupby("dex", as_index=False)["raw_count"].mean().rename(columns={"raw_count": "mean_raw_count"})
    return (
        f"Gene: {row['gene_id']} / {row['symbol']} / {row['gene_name']}\n\n"
        f"Mean raw counts by dex group:\n{means.to_string(index=False)}\n\n"
        f"Per-sample raw counts:\n{per_sample.to_string(index=False)}"
    )

@tool
def retrieve_airway_documents(query: str) -> str:
    """Retrieve relevant text from docs/paper.pdf plus data/README.md and data/DESeq2_results_airway_readme.md. Use this for protocol, paper background, dataset provenance, DESeq2 workflow notes, and biological context."""
    if doc_vector_store is not None:
        docs = doc_vector_store.similarity_search(query, k=5)
    else:
        terms = [term.strip(".,:;()[]{}").lower() for term in query.split() if len(term.strip(".,:;()[]{}")) > 2]
        docs = sorted(
            doc_chunks,
            key=lambda doc: sum(doc.page_content.lower().count(term) for term in terms),
            reverse=True,
        )[:5]
    return "\n\n".join(
        f"[{doc.metadata.get('source')} | {doc.metadata.get('doc_type')} | page {doc.metadata.get('page', 'n/a')}]\n{doc.page_content[:1200]}"
        for doc in docs
    )

print(f"Loaded {len(sample_metadata)} samples, {len(gene_counts)} count rows, {len(deseq2_results)} DESeq2 rows, and {len(doc_chunks)} document chunks ({document_retrieval_mode}).")


## 1. Plain LLM

Plain LLM architecture sends the user question directly to the model, so any answer depends on model priors rather than external data. We explored this in notebook 01.


In [ ]:
question = "Which genes are most significantly upregulated by dexamethasone in the local airway RNA-seq dataset?"
model = init_chat_model("gemini-2.5-flash", model_provider="google_genai", temperature=0)
response = model.invoke(question)

df = pd.DataFrame([
    {"architecture": "plain_llm", "question": question, "answer": response.content},
])
df.style.set_properties(
    subset=["answer"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


## 2. LLM + RAG

LLM plus RAG retrieves relevant context before generation, grounding the response in supplied text documents without giving the model autonomous tool control.


In [ ]:
question = "What does the paper and local documentation say about dexamethasone effects in airway smooth muscle cells?"
context = retrieve_airway_documents.invoke({"query": "dexamethasone airway smooth muscle cells CRISPLD2 glucocorticoid RNA-seq"})
prompt = f"Use only this retrieved local context to answer.\n\n{context}\n\nQuestion: {question}"

model = init_chat_model("gemini-2.5-flash", model_provider="google_genai", temperature=0)
response = model.invoke(prompt)

df = pd.DataFrame([
    {"architecture": "llm_plus_rag", "question": question, "answer": response.content, "context": context},
])
df.style.set_properties(
    subset=["answer"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


## 3. ReAct agent

ReAct-style agents iterate between deciding an action, using a tool, observing the result, and answering; inspect workflow-level tool calls rather than hidden reasoning. This opens a plethora of abilites as a tool can be anything programmatic.


In [ ]:
model = init_chat_model("gemini-2.5-flash", model_provider="google_genai", temperature=0.1)
react_like_prompt = "Use the airway tools step by step. Query structured CSV data with SQL tools, retrieve paper/readme context with the document tool, and do not invent dataset facts."
agent = create_agent(model, [query_sample_metadata, query_deseq2_results, query_gene_counts, summarize_gene_counts, retrieve_airway_documents], system_prompt=react_like_prompt)

for event in agent.stream(
    {"messages": [{"role": "user", "content": "How many dexamethasone-treated samples are present, and which significantly upregulated genes best support a dexamethasone response?"}]},
    stream_mode="values",
):
    event["messages"][-1].pretty_print()


## 4. Tool-calling agent

Tool-calling agents use provider-native tool calls, which usually produces cleaner executable calls and fewer hallucinated tool names than text-only action formats.


In [ ]:
model = init_chat_model("gemini-2.5-flash", model_provider="google_genai", temperature=0)
tool_agent = create_agent(
    model,
    [query_sample_metadata, query_deseq2_results, summarize_gene_counts, retrieve_airway_documents],
    system_prompt="Use SQL for sample metadata and DESeq2 statistics, summarize_gene_counts for raw counts, and retrieve_airway_documents for paper/readme context.",
)

for event in tool_agent.stream(
    {"messages": [{"role": "user", "content": "Report the dex treatment group counts, the DESeq2 result for CRISPLD2, and the raw CRISPLD2 counts by treatment group."}]},
    stream_mode="values",
):
    event["messages"][-1].pretty_print()


## 5. LangGraph workflow

LangGraph is a more modern version of LangChain agents workflow in which the control flow is more explicit, improving reproducibility, debugging, and deterministic execution.


In [ ]:
class GraphState(TypedDict):
    question: str
    structured_context: str
    document_context: str
    answer: str

model = init_chat_model("gemini-2.5-flash", model_provider="google_genai", temperature=0)

def retrieve_structured_node(state: GraphState):
    return {
        "structured_context": query_deseq2_results.invoke({
            "sql": "select gene_id, symbol, log2FoldChange, padj from deseq2_results where padj < 0.05 order by log2FoldChange desc limit 10"
        })
    }

def retrieve_documents_node(state: GraphState):
    return {"document_context": retrieve_airway_documents.invoke({"query": state["question"]})}

def answer_node(state: GraphState):
    prompt = f"Structured context:\n{state['structured_context']}\n\nDocument context:\n{state['document_context']}\n\nQuestion: {state['question']}"
    response = model.invoke(prompt)
    return {"answer": response.content}

graph = StateGraph(GraphState)
graph.add_node("retrieve_structured", retrieve_structured_node)
graph.add_node("retrieve_documents", retrieve_documents_node)
graph.add_node("answer", answer_node)
graph.add_edge(START, "retrieve_structured")
graph.add_edge("retrieve_structured", "retrieve_documents")
graph.add_edge("retrieve_documents", "answer")
graph.add_edge("answer", END)
workflow = graph.compile()
result = workflow.invoke({"question": "Which local DESeq2 genes are strongest dexamethasone-induced candidates, and how does the documentation frame the experiment?", "structured_context": "", "document_context": "", "answer": ""})

df = pd.DataFrame([
    {"architecture": "langgraph_workflow", "question": result["question"], "answer": result["answer"], "structured_context": result["structured_context"], "document_context": result["document_context"]},
])
df.style.set_properties(
    subset=["answer"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


## 6. Multi-agent

Multi-agent architecture assigns specialized roles to separate agents, which can improve modularity but adds coordination cost compared with one general agent.


In [ ]:
model = init_chat_model("gemini-2.5-flash", model_provider="google_genai", temperature=0)
researcher = create_agent(model, [retrieve_airway_documents], system_prompt="You are the paper and documentation researcher. Return only notes grounded in retrieved local documents.")
statistician = create_agent(model, [query_sample_metadata, query_deseq2_results], system_prompt="You answer sample and DESeq2 questions using SQL over the local CSV-derived tables.")
expression_agent = create_agent(model, [summarize_gene_counts, query_gene_counts], system_prompt="You answer raw count questions using the local airway count matrix.")
summarizer = create_agent(model, [], system_prompt="Synthesize upstream agent outputs into a concise final answer with no unsupported claims.")

question = "What does the local airway dataset show about CRISPLD2 under dexamethasone treatment?"
research_notes = researcher.invoke({"messages": [{"role": "user", "content": "Retrieve local context about CRISPLD2, dexamethasone, and airway smooth muscle cells."}]})
stat_notes = statistician.invoke({"messages": [{"role": "user", "content": "Find the DESeq2 result for CRISPLD2 and count samples by dex group."}]})
count_notes = expression_agent.invoke({"messages": [{"role": "user", "content": "Summarize raw counts for CRISPLD2 by dex treatment group."}]})
summary_prompt = f"Researcher:\n{research_notes['messages'][-1].content}\n\nStatistician:\n{stat_notes['messages'][-1].content}\n\nRaw counts:\n{count_notes['messages'][-1].content}"
final = summarizer.invoke({"messages": [{"role": "user", "content": summary_prompt}]})

df = pd.DataFrame([
    {"architecture": "multi_agent", "question": question, "answer": final["messages"][-1].content},
])
df.style.set_properties(
    subset=["answer"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


## 7. Router agent

Router agents classify the question and dispatch it to SQL, RAG, or Python, which can outperform one general agent when tool choice is predictable.


In [ ]:
questions = [
    "How many dexamethasone-treated samples are in airway_metadata.csv?",
    "What are the top significant DESeq2 genes by positive log2FoldChange?",
    "What are the raw counts for CRISPLD2?",
    "What does the paper say about glucocorticoid response in airway smooth muscle?",
]

rows = []
for question in questions:
    q = question.lower()
    if "how many" in q or "metadata" in q or "samples" in q:
        destination = "sample_metadata_sql"
        answer = query_sample_metadata.invoke({"sql": "select dex, count(*) as n from sample_metadata group by dex"})
    elif "deseq2" in q or "log2foldchange" in q or "significant" in q:
        destination = "deseq2_sql"
        answer = query_deseq2_results.invoke({"sql": "select gene_id, symbol, log2FoldChange, padj from deseq2_results where padj < 0.05 order by log2FoldChange desc limit 10"})
    elif "raw counts" in q or "counts" in q:
        destination = "count_matrix_tool"
        answer = summarize_gene_counts.invoke({"identifier": "CRISPLD2"})
    else:
        destination = "document_retrieval"
        answer = retrieve_airway_documents.invoke({"query": question})
    rows.append({"question": question, "destination": destination, "answer": str(answer)})

df = pd.DataFrame(rows)
df.style.set_properties(
    subset=["answer"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


## 8. Sequential chains

Sequential chains run a fixed pipeline such as prompt, retriever, summarizer, and formatter, which is easier to reproduce than a flexible agent loop.


In [ ]:
question = "Summarize sample composition, top dexamethasone-upregulated DESeq2 genes, and paper context as a compact table."
sample_context = query_sample_metadata.invoke({"sql": "select dex, count(*) as n from sample_metadata group by dex"})
deg_context = query_deseq2_results.invoke({"sql": "select gene_id, symbol, log2FoldChange, padj from deseq2_results where padj < 0.05 order by log2FoldChange desc limit 8"})
document_context = retrieve_airway_documents.invoke({"query": "dexamethasone airway smooth muscle RNA-seq DESeq2 glucocorticoid response"})
model = init_chat_model("gemini-2.5-flash", model_provider="google_genai", temperature=0)

response = model.invoke(f"Summarize these local data contexts:\nSamples:\n{sample_context}\n\nDESeq2:\n{deg_context}\n\nDocuments:\n{document_context}")
summary = response.content
response = model.invoke(f"Format as a markdown table with columns evidence_type, local_source, key_finding:\n{summary}")

df = pd.DataFrame([
    {"architecture": "sequential_chain", "question": question, "answer": response.content},
])
df.style.set_properties(
    subset=["answer"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


## 9. Memory techniques


### i. No memory

No memory means each question is independent, so follow-up questions lack previous conversational context.


In [ ]:
model = init_chat_model("gemini-2.5-flash", model_provider="google_genai", temperature=0)
first_response = model.invoke("The local dataset is the airway RNA-seq dataset comparing dexamethasone-treated and untreated human airway smooth muscle cell samples.")
followup_response = model.invoke("What treatment comparison is the local dataset about?")

df = pd.DataFrame([
    {"turn": "first", "question": "The local dataset is the airway RNA-seq dataset comparing dexamethasone-treated and untreated human airway smooth muscle cell samples.", "answer": first_response.content},
    {"turn": "followup_without_memory", "question": "What treatment comparison is the local dataset about?", "answer": followup_response.content},
])
df.style.set_properties(
    subset=["answer"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


### ii. Conversation memory

Conversation memory retains previous messages, allowing follow-up questions to resolve references and continue the same topic.


In [ ]:
messages = [
    {"role": "user", "content": "The local dataset compares dexamethasone-treated and untreated airway smooth muscle cell samples."},
    {"role": "assistant", "content": "Noted."},
    {"role": "user", "content": "What treatment comparison is the local dataset about?"},
]
model = init_chat_model("gemini-2.5-flash", model_provider="google_genai", temperature=0)
response = model.invoke(messages)

df = pd.DataFrame([
    {"memory": "conversation", "question": messages[-1]["content"], "answer": response.content},
])
df.style.set_properties(
    subset=["answer"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


### iii. Windowed memory

Windowed memory keeps only the most recent turns, trading context retention for lower token cost and fewer stale details.


In [ ]:
full_history = [
    {"role": "user", "content": "The local files include airway_counts.csv, airway_metadata.csv, DESeq2_results_airway.csv, readmes, and paper.pdf."},
    {"role": "assistant", "content": "Noted."},
    {"role": "user", "content": "The key treatment variable is the dex column with trt and untrt groups."},
    {"role": "assistant", "content": "Noted."},
    {"role": "user", "content": "Which local files and variable define the treatment comparison?"},
]
model = init_chat_model("gemini-2.5-flash", model_provider="google_genai", temperature=0)

short_window_response = model.invoke(full_history[-3:])
full_conversation_response = model.invoke(full_history)

df = pd.DataFrame([
    {"memory": "short_window", "question": full_history[-1]["content"], "answer": short_window_response.content},
    {"memory": "full_conversation", "question": full_history[-1]["content"], "answer": full_conversation_response.content},
])
df.style.set_properties(
    subset=["answer"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


## 10. Tool usage


### i. No tools

No tools means the LLM cannot compute, query databases, or retrieve data, so it may guess when asked for exact results.


In [ ]:
model = init_chat_model("gemini-2.5-flash", model_provider="google_genai", temperature=0)
question = "What is the DESeq2 log2FoldChange and adjusted p-value for CRISPLD2 in the local airway dataset?"
response = model.invoke(question)

df = pd.DataFrame([
    {"tool_access": "no_tools", "question": question, "answer": response.content},
])
df.style.set_properties(
    subset=["answer"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


### ii. Python tool

Python tools allow exact computation, which is useful for numeric tasks such as expression summaries.


In [ ]:
model = init_chat_model("gemini-2.5-flash", model_provider="google_genai", temperature=0)
agent = create_agent(model, [summarize_gene_counts], system_prompt="Use the raw count summary tool for gene-level expression summaries from airway_counts.csv.")

for event in agent.stream(
    {"messages": [{"role": "user", "content": "Summarize raw CRISPLD2 counts by dex treatment group."}]},
    stream_mode="values",
):
    event["messages"][-1].pretty_print()


### iii. SQL tool

SQL tools allow the agent to query structured data, reducing hallucination on database questions such as sample counts.


In [ ]:
model = init_chat_model("gemini-2.5-flash", model_provider="google_genai", temperature=0)
agent = create_agent(model, [query_sample_metadata, query_deseq2_results, query_gene_counts], system_prompt="Use SQL for questions about the local airway CSV tables.")

for event in agent.stream(
    {"messages": [{"role": "user", "content": "How many samples are treated versus untreated, and what is the CRISPLD2 DESeq2 result?"}]},
    stream_mode="values",
):
    event["messages"][-1].pretty_print()


### iv. Multiple tools

Multiple tools let the agent choose among Python, SQL, and retrieval, so tool selection behavior becomes part of the benchmark.


In [ ]:
model = init_chat_model("gemini-2.5-flash", model_provider="google_genai", temperature=0)
agent = create_agent(
    model,
    [query_sample_metadata, query_deseq2_results, query_gene_counts, summarize_gene_counts, retrieve_airway_documents],
    system_prompt="Choose the most specific local airway tool before answering. Use SQL for tables, summarize_gene_counts for raw count summaries, and retrieve_airway_documents for paper/readme context.",
)

for event in agent.stream(
    {"messages": [{"role": "user", "content": "Combine sample counts, top upregulated DESeq2 genes, CRISPLD2 raw counts, and paper context into one answer."}]},
    stream_mode="values",
):
    event["messages"][-1].pretty_print()


### v. Tool descriptions

Tool descriptions tell the model when and how to use each tool, so vague descriptions can reduce tool selection accuracy.


In [ ]:
@tool
def vague_airway_data_tool(query: str) -> str:
    """Search airway data."""
    return query_sample_metadata.invoke({"sql": "select dex, count(*) as n from sample_metadata group by dex"})

@tool
def clear_sample_count_tool(query: str) -> str:
    """Return exact dexamethasone treatment-group sample counts from the local airway_metadata.csv-derived sample_metadata SQL table."""
    return query_sample_metadata.invoke({"sql": "select dex, count(*) as n from sample_metadata group by dex"})

model = init_chat_model("gemini-2.5-flash", model_provider="google_genai", temperature=0)

print("\n### poor_description")
agent = create_agent(model, [vague_airway_data_tool], system_prompt="Use a tool for exact local airway sample counts.")
for event in agent.stream({"messages": [{"role": "user", "content": "How many dexamethasone-treated and untreated samples are there?"}]}, stream_mode="values"):
    event["messages"][-1].pretty_print()

print("\n### good_description")
agent = create_agent(model, [clear_sample_count_tool], system_prompt="Use a tool for exact local airway sample counts.")
for event in agent.stream({"messages": [{"role": "user", "content": "How many dexamethasone-treated and untreated samples are there?"}]}, stream_mode="values"):
    event["messages"][-1].pretty_print()
